In [1]:
# ============================================================
# CELL 1 — IMPORTS, CONFIGURATION & BACKUP
# ============================================================
# ALGORITHM NOTE
# ─────────────────────────────────────────────────────────────
# Your previous notebook used the default algorithm parameter.
# sklearn < 1.2  → default = SAMME.R (real-valued probabilities,
#                  faster convergence, usually better accuracy)
# sklearn ≥ 1.4  → SAMME.R removed; only SAMME is available.
# Since you get an error on algorithm='SAMME.R', you are on
# sklearn ≥ 1.4 and both notebooks effectively run SAMME.
# SAMME uses discrete class predictions, not probabilities.
# ─────────────────────────────────────────────────────────────

import os
import shutil
import datetime
import numpy as np
import joblib

from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA PATHS ────────────────────────────────────────────────
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ───────────────────────────────────────────────
MODEL_SAVE_PATH  = r"C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost_mlp.joblib"
MODEL_BACKUP_DIR = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

RANDOM_STATE = 42
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== ADABOOST + MLP WEAK-LEARNER CONFIGURATION ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

if os.path.exists(MODEL_SAVE_PATH):
    timestamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(
        MODEL_BACKUP_DIR,
        f"tissue_density_model_adaboost_mlp_{timestamp}.joblib"
    )
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f" Existing model backed up → {backup_path}")
else:
    print("No existing model found — fresh training run.")

=== ADABOOST + MLP WEAK-LEARNER CONFIGURATION ===
Feature file : C:\Users\kalig\OneDrive\Desktop\train_X.npy
Label file   : C:\Users\kalig\OneDrive\Desktop\train_y.npy
Model output : C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost_mlp.joblib
 Existing model backed up → C:\Users\kalig\OneDrive\Desktop\model_backups\tissue_density_model_adaboost_mlp_20260611_222252.joblib


In [2]:
# ============================================================
# CELL 2 — LOAD, SPLIT & BALANCE DATA
# (identical pipeline to the decision-stump notebook)
# ============================================================
print("\nLoading feature matrix and labels...")
X = np.load(X_PATH)
y = np.load(Y_PATH)

print(f"Loaded → X={X.shape}, y={y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# Split first — test set stays completely untouched by SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nBefore SMOTE — training distribution: {Counter(y_train)}")

smote = SMOTE(
    sampling_strategy='auto',
    random_state=RANDOM_STATE,
    k_neighbors=7
)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After  SMOTE — training distribution: {Counter(y_train_balanced)}")

real_train_images  = X_train.shape[0]
total_train_images = X_train_balanced.shape[0]
synthetic_images   = total_train_images - real_train_images
test_images        = X_test.shape[0]

print(f"\n{'='*52}")
print("DATA SUMMARY")
print(f"{'='*52}")
print(f"Original training images : {real_train_images}")
print(f"Synthetic images added   : {synthetic_images}")
print(f"Total training images    : {total_train_images}")
print(f"Test images              : {test_images}")
print(f"Features per image       : {X_train_balanced.shape[1]}")
print(f"{'='*52}")


Loading feature matrix and labels...
Loaded → X=(9999, 1792), y=(9999,)
Full dataset class distribution: {np.int64(1): np.int64(1172), np.int64(2): np.int64(4282), np.int64(3): np.int64(3991), np.int64(4): np.int64(554)}

Before SMOTE — training distribution: Counter({np.int64(2): 3425, np.int64(3): 3193, np.int64(1): 938, np.int64(4): 443})
After  SMOTE — training distribution: Counter({np.int64(1): 3425, np.int64(2): 3425, np.int64(3): 3425, np.int64(4): 3425})

DATA SUMMARY
Original training images : 7999
Synthetic images added   : 5701
Total training images    : 13700
Test images              : 2000
Features per image       : 1792


In [3]:
# ============================================================
# CELL 3 — MLP WEAK-LEARNER WRAPPER
# ============================================================
# WHY A WRAPPER?
# ─────────────────────────────────────────────────────────────
# AdaBoostClassifier internally calls  estimator.fit(X, y,
# sample_weight=w)  to re-focus each learner on hard samples.
# sklearn's MLPClassifier.fit() does NOT accept sample_weight,
# so a direct plug-in raises a TypeError.
#
# Solution: convert the weight vector into a resampling
# distribution — draw len(X) indices with replacement using
# the normalised weights as probabilities, then train the MLP
# on that resampled subset.  This exactly mirrors what Bagging
# does and is a standard workaround for weightless estimators.
# ─────────────────────────────────────────────────────────────
#
# ARCHITECTURE GUIDE (pick ONE — see Cell 4)
# ─────────────────────────────────────────────────────────────
#  OPTION A — ultra-weak  (16,)        1 hidden layer, 16 nodes
#             Most similar in power to a depth-1 stump.
#             Safest choice if you suspect overfitting.
#
#  OPTION B — weak        (32,)        1 hidden layer, 32 nodes
#             Good balance: meaningful non-linear patterns,
#             still clearly a weak learner.  RECOMMENDED START.
#
#  OPTION C — mild        (32, 16)     2 hidden layers
#             Slightly stronger; useful if B plateaus.
#             Watch validation accuracy for signs of overfit.
# ─────────────────────────────────────────────────────────────

class WeakMLP(BaseEstimator, ClassifierMixin):
    """
    A tiny MLP wrapper compatible with AdaBoostClassifier.

    Parameters
    ----------
    hidden_layer_sizes : tuple
        Architecture.  Keep small — this must be a *weak* learner.
    max_iter : int
        Max SGD epochs per estimator.  Low values (50-100) prevent
        the MLP from fully converging, which is intentional.
    activation : str
        'relu' (default) or 'tanh'.  relu trains faster.
    random_state : int or None
        Passed through to MLPClassifier for reproducibility.
    """

    def __init__(
        self,
        hidden_layer_sizes=(32,),
        max_iter=50,
        activation='relu',
        random_state=None
    ):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.max_iter           = max_iter
        self.activation         = activation
        self.random_state       = random_state

    # ----------------------------------------------------------
    def fit(self, X, y, sample_weight=None):
        rng = np.random.RandomState(self.random_state)

        if sample_weight is not None:
            # Normalise so weights sum to 1 (valid probability dist)
            w   = np.asarray(sample_weight, dtype=float)
            w  /= w.sum()
            idx = rng.choice(len(X), size=len(X), replace=True, p=w)
            X, y = X[idx], y[idx]

        self.mlp_ = MLPClassifier(
            hidden_layer_sizes = self.hidden_layer_sizes,
            max_iter           = self.max_iter,
            activation         = self.activation,
            solver             = 'adam',
            random_state       = self.random_state,
            early_stopping     = False,   # keep weak — no early stop
            warm_start         = False,
        )
        self.mlp_.fit(X, y)
        self.classes_ = self.mlp_.classes_   # required by AdaBoost
        return self

    # ----------------------------------------------------------
    def predict(self, X):
        check_is_fitted(self, 'mlp_')
        return self.mlp_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self, 'mlp_')
        return self.mlp_.predict_proba(X)

print("WeakMLP wrapper defined.")
print("Architecture options:")
print("  Option A — ultra-weak : hidden_layer_sizes=(16,)")
print("  Option B — weak (REC) : hidden_layer_sizes=(32,)")
print("  Option C — mild       : hidden_layer_sizes=(32, 16)")

WeakMLP wrapper defined.
Architecture options:
  Option A — ultra-weak : hidden_layer_sizes=(16,)
  Option B — weak (REC) : hidden_layer_sizes=(32,)
  Option C — mild       : hidden_layer_sizes=(32, 16)


In [4]:
# ============================================================
# CELL 4 — BUILD & TRAIN ADABOOST WITH MLP WEAK LEARNERS
# ============================================================
# ── ARCHITECTURE SELECTION ───────────────────────────────────
#  Change ARCH to 'A', 'B', or 'C' to switch between options.
ARCH = 'C'   # <── EDIT THIS

ARCH_MAP = {
    'A': {'hidden_layer_sizes': (16,),     'label': 'Ultra-weak (16,)'},
    'B': {'hidden_layer_sizes': (32,),     'label': 'Weak (32,)  ← recommended'},
    'C': {'hidden_layer_sizes': (32, 16),  'label': 'Mild (32, 16)'},
}

chosen     = ARCH_MAP[ARCH]
arch_sizes = chosen['hidden_layer_sizes']

print(f"Selected architecture : {chosen['label']}")

# ── ADABOOST HYPERPARAMETERS ─────────────────────────────────
# n_estimators is lower than the stump version (300) because
# each MLP contributes more signal per round. 150 is a good
# starting point; reduce to 100 if training is too slow.
#
# learning_rate controls the shrinkage applied to each weak
# learner's contribution. 0.8 works well; lower (0.5) if you
# see training accuracy diverge from validation accuracy.

N_ESTIMATORS  = 150
LEARNING_RATE = 0.8
MLP_MAX_ITER  = 50    # keep low → deliberate weak learner

base_estimator = WeakMLP(
    hidden_layer_sizes = arch_sizes,
    max_iter           = MLP_MAX_ITER,
    activation         = 'relu',
    random_state       = RANDOM_STATE
)

model = AdaBoostClassifier(
    estimator     = base_estimator,
    n_estimators  = N_ESTIMATORS,
    learning_rate = LEARNING_RATE,
    random_state  = RANDOM_STATE
    # algorithm is not specified — on sklearn ≥ 1.4 this is SAMME
    # (SAMME.R was removed; specifying it raises an error)
)

print(f"\nAdaBoost config:")
print(f"  n_estimators  = {N_ESTIMATORS}")
print(f"  learning_rate = {LEARNING_RATE}")
print(f"  MLP max_iter  = {MLP_MAX_ITER}  (intentionally low)")
print(f"  Algorithm     = SAMME (only option on sklearn ≥ 1.4)")
print("\nTraining… (this will take longer than decision stumps)")

model.fit(X_train_balanced, y_train_balanced)
print(" Training complete!")

Selected architecture : Mild (32, 16)

AdaBoost config:
  n_estimators  = 150
  learning_rate = 0.8
  MLP max_iter  = 50  (intentionally low)
  Algorithm     = SAMME (only option on sklearn ≥ 1.4)

Training… (this will take longer than decision stumps)


c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimiza

 Training complete!


c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


In [5]:
# ============================================================
# CELL 5 — EVALUATE MODEL
# ============================================================
print("\nEvaluating on held-out test set...")

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f"\n{'='*52}")
print("PERFORMANCE REPORT")
print(f"{'='*52}")
print(f"Architecture            : {chosen['label']}")
print(f"Images model trained on : {total_train_images}")
print(f" ↳ Real images          : {real_train_images}")
print(f" ↳ SMOTE synthetic      : {synthetic_images}")
print(f"Images tested on        : {test_images}")
print(f"Overall accuracy        : {acc:.4f}")
print(f"{'='*52}")

print("\nPer-class breakdown:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        'Tissue Density Category 1',
        'Tissue Density Category 2',
        'Tissue Density Category 3',
        'Tissue Density Category 4'
    ],
    zero_division=0
))

print("Confusion Matrix:")
header = "              " + "  ".join(f"Pred {i+1}" for i in range(4))
print(header)
for i, row in enumerate(cm):
    print(f"Actual {i+1}     " + "  ".join(f"{v:6d}" for v in row))

print("\nFirst 5 class-probability rows (softmax from MLP):")
print(y_proba[:5])


Evaluating on held-out test set...

PERFORMANCE REPORT
Architecture            : Mild (32, 16)
Images model trained on : 13700
 ↳ Real images          : 7999
 ↳ SMOTE synthetic      : 5701
Images tested on        : 2000
Overall accuracy        : 0.6830

Per-class breakdown:
                           precision    recall  f1-score   support

Tissue Density Category 1       0.52      0.55      0.54       234
Tissue Density Category 2       0.70      0.70      0.70       857
Tissue Density Category 3       0.75      0.73      0.74       798
Tissue Density Category 4       0.43      0.46      0.44       111

                 accuracy                           0.68      2000
                macro avg       0.60      0.61      0.61      2000
             weighted avg       0.69      0.68      0.68      2000

Confusion Matrix:
              Pred 1  Pred 2  Pred 3  Pred 4
Actual 1        128     105       1       0
Actual 2        112     602     140       3
Actual 3          4     144     58

In [6]:
# ============================================================
# CELL 6 — CLASS CONFIDENCE SNAPSHOT
# ============================================================
print("\nSample class-probability breakdown (first 10 test images):\n")

header = (
    f"{'Image':<8} | "
    f"{'Cat 1':>6} {'Cat 2':>6} {'Cat 3':>6} {'Cat 4':>6} | "
    f"{'Pred':>5} {'True':>5}"
)
print(header)
print("-" * len(header))

for i in range(min(10, len(X_test))):
    p_row = y_proba[i]
    pred  = y_pred[i]
    true  = y_test[i]
    match = "✅" if pred == true else "❌"

    print(
        f"{i+1:<8} | "
        f"{p_row[0]:>6.3f} {p_row[1]:>6.3f} {p_row[2]:>6.3f} {p_row[3]:>6.3f} | "
        f"{pred:>5} {true:>5}  {match}"
    )


Sample class-probability breakdown (first 10 test images):

Image    |  Cat 1  Cat 2  Cat 3  Cat 4 |  Pred  True
----------------------------------------------------
1        |  0.223  0.261  0.283  0.233 |     3     3  ✅
2        |  0.223  0.240  0.288  0.249 |     3     3  ✅
3        |  0.222  0.223  0.269  0.285 |     4     3  ❌
4        |  0.222  0.283  0.272  0.222 |     2     2  ✅
5        |  0.222  0.222  0.283  0.273 |     3     3  ✅
6        |  0.256  0.287  0.235  0.223 |     2     2  ✅
7        |  0.223  0.279  0.276  0.222 |     2     2  ✅
8        |  0.222  0.260  0.290  0.227 |     3     3  ✅
9        |  0.238  0.271  0.269  0.223 |     2     2  ✅
10       |  0.223  0.231  0.280  0.266 |     3     4  ❌


In [7]:
# ============================================================
# CELL 7 — SAVE MODEL
# ============================================================
print(f"\nSaving model → {MODEL_SAVE_PATH}")
joblib.dump(model, MODEL_SAVE_PATH)
print(" Done! AdaBoost + MLP tissue density model saved.")


Saving model → C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost_mlp.joblib
 Done! AdaBoost + MLP tissue density model saved.


In [8]:
# ============================================================
# CELL 8 — PREDICTION UTILITY
# ============================================================
def predict_new(X_new: np.ndarray):
    """
    Run inference on new feature vectors.

    Parameters
    ----------
    X_new : np.ndarray, shape (n_samples, n_features)

    Returns
    -------
    preds       : predicted tissue density class labels
    class_probs : per-class probabilities from the MLP ensemble
    """
    preds       = model.predict(X_new)
    class_probs = model.predict_proba(X_new)
    return preds, class_probs

print(" Prediction utility ready.")
print("Usage:")
print("  preds, class_probs = predict_new(X_new)")
print("\nOr load from disk:")
print(f"  model = joblib.load(r'{MODEL_SAVE_PATH}')")
print("  preds, class_probs = model.predict(X_new), model.predict_proba(X_new)")

 Prediction utility ready.
Usage:
  preds, class_probs = predict_new(X_new)

Or load from disk:
  model = joblib.load(r'C:\Users\kalig\OneDrive\Desktop\tissue_density_model_adaboost_mlp.joblib')
  preds, class_probs = model.predict(X_new), model.predict_proba(X_new)
